# Generating Masks from Images and Labels

## Renaming files

When I imported the images and labels from Roboflow, it changed the file names, I will have to rename them

In [ ]:
import os
import re

def rename_files(folder, ext, dry_run=True):
    for filename in os.listdir(folder):
        # Match pattern like "1_1_1_jpg.rf.<hash>.ext"
        match = re.match(r"(\d+_\d+_\d+)_jpg\.rf\.[a-f0-9]+\.{}".format(ext), filename)
        if match:
            new_name = f"{match.group(1)}.{ext}"
            old_path = os.path.join(folder, filename)
            new_path = os.path.join(folder, new_name)
            if dry_run:
                print(f"Would rename: {old_path} -> {new_path}")
            else:
                os.rename(old_path, new_path)

In [ ]:
# Rename images
rename_files('../data/sorghum/labels', 'txt', dry_run=False)

For wheat, the names were different, being spikeId_collection. Need to change

In [12]:
def rename_spike_files(folder, ext, dry_run=True):
    pattern = re.compile(r"(\d+)_(\d+)_jpg\.rf\.[a-f0-9]+\.{}".format(ext))
    renames = []
    file_list = os.listdir(folder)
    for fname in file_list:
        match = pattern.match(fname)
        if match:
            spike_id = int(match.group(1))
            collection = match.group(2)
            plot_number = ((spike_id - 1) // 10) + 1
            spike_number = ((spike_id - 1) % 10) + 1
            new_name = f"{collection}_{plot_number}_{spike_number}.{ext}"
            old_path = os.path.join(folder, fname)
            new_path = os.path.join(folder, new_name)
            renames.append((old_path, new_path))
    if dry_run:
        print("Dry run (no files will be renamed):")
        for old, new in renames:
            print(f"{old} -> {new}")
    else:
        print("Renaming files:")
        for old, new in renames:
            print(f"{old} -> {new}")
            os.rename(old, new)

In [ ]:
rename_spike_files('../data/wheat/images', 'jpg', dry_run=False)

## Generating Masks

The labels were made using Yolov8 label format, need to use it to generate binary masks and save them in the another folder with the same name

In [3]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def create_masks_from_yolov8(images_folder, labels_folder, masks_folder, img_ext='jpg', label_ext='txt', show_example=True):
    os.makedirs(masks_folder, exist_ok=True)
    image_files = [f for f in os.listdir(images_folder) if f.endswith(f'.{img_ext}')]
    for img_file in image_files:
        img_path = os.path.join(images_folder, img_file)
        label_file = img_file.replace(f'.{img_ext}', f'.{label_ext}')
        label_path = os.path.join(labels_folder, label_file)
        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 2:
                        continue
                    # Yolov8 format: class_id x_center y_center width height [polygon points...]
                    if len(parts) > 5:
                        # polygon mask (segmentation)
                        pts = np.array(parts[1:], dtype=float).reshape(-1, 2)
                        pts[:, 0] *= w
                        pts[:, 1] *= h
                        pts = pts.astype(np.int32)
                        cv2.fillPoly(mask, [pts], 255)
                    else:
                        # bbox mask
                        _, x, y, bw, bh = map(float, parts[:5])
                        x1 = int((x - bw/2) * w)
                        y1 = int((y - bh/2) * h)
                        x2 = int((x + bw/2) * w)
                        y2 = int((y + bh/2) * h)
                        cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)
        # Show example
        if show_example:
            plt.figure(figsize=(10,5))
            plt.subplot(1,2,1)
            plt.title('Image')
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.subplot(1,2,2)
            plt.title('Mask')
            plt.imshow(mask, cmap='gray')
            plt.axis('off')
            plt.show()
            break
        else:
            mask_path = os.path.join(masks_folder, img_file.replace(f'.{img_ext}', '.png'))
            cv2.imwrite(mask_path, mask)

In [6]:
create_masks_from_yolov8('../data/wheat/images', '../data/wheat/labels', '../data/wheat/masks', show_example=False)

## Creating METADATA

Creating a csv file containing information about the crop, collection date, genotype, individual for each image we have

In [7]:
import pandas as pd
import glob
import os

data = []
for crop in ['wheat', 'sorghum']:
    img_folder = f'../data/{crop}/images'
    img_files = glob.glob(os.path.join(img_folder, '*.jpg'))
    for img_path in img_files:
        img_name = os.path.splitext(os.path.basename(img_path))[0]
        parts = img_name.split('_')
        if len(parts) == 3:
            collection_date, genotype, individual = parts
            data.append({
                'crop': crop,
                'collection_date': collection_date,
                'genotype': genotype,
                'individual': individual,
                'image_path': img_path
            })

df_images = pd.DataFrame(data)
print(df_images.head())

# Save table
df_images.to_csv('../data/images_metadata.csv', index=False)

    crop collection_date genotype individual                        image_path
0  wheat               1       10          1   ../data/wheat/images\1_10_1.jpg
1  wheat               1       10         10  ../data/wheat/images\1_10_10.jpg
2  wheat               1       10          2   ../data/wheat/images\1_10_2.jpg
3  wheat               1       10          3   ../data/wheat/images\1_10_3.jpg
4  wheat               1       10          4   ../data/wheat/images\1_10_4.jpg
